# ⑤ 평가 — 네 가지로 나눠 채점
담당: 평가 · 코드: `rag/evaluation/evaluation.py`

04 생성의 `artifacts/answers.jsonl` 하나를 네 가지로 채점해, 나쁜 답이 검색에서 왔는지 · 생성에서 왔는지 가릅니다.
심판은 아래 설정 셀의 `JUDGE` 로 고릅니다. 각 표의 마지막 행(`평균`)이 요약입니다.
- `"local"` (기본): 로컬 KURE-v1 임베딩과 02 의 리랭커(크로스 인코더)로 같은 이름의 지표를 계산합니다. **API 를 부르지 않아 비용 0원**이고 GPU 에서 몇 분이면 끝납니다.
- `"ragas"`: RAGAS + gpt 심판(OpenAI API, 유료). 두 방식은 같은 지표 이름이어도 값의 크기가 달라 서로 비교하지 않습니다.
04 를 Langfuse 키와 함께 실행했다면 질문별 점수가 그 질문의 Langfuse 추적에도 붙습니다 (지연 시간 · 토큰 · 비용은 추적에 이미 있어 점수로 보내지 않음).

`"ragas"` 점수는 16문항씩 채점할 때마다 `artifacts/checkpoints/judge/<평가>.jsonl` 에 저장합니다. 중간에 끊기면 그 평가 셀을 다시 실행하세요 — 남은 문항만 채점합니다.
답 · 근거 · 정답이나 심판 설정이 바뀐 문항은 다시 채점하고, NaN(심판 호출 실패 포함)은 저장하지 않아 다음 실행에서 다시 채점합니다.

| 평가 | 판단하는 것 | 저장 파일 |
|---|---|---|
| Retrieval | 검색한 청크가 정답에 필요한 내용을 담았나 | `artifacts/retrieval.csv` |
| Generation | 답이 질문에 맞고 핵심 사실을 담았나 | `artifacts/generation.csv` |
| Grounding | 답이 검색한 청크 안에서만 나왔나 (환각) | `artifacts/grounding.csv` |
| End-to-End | 최종 답이 정답과 맞나 · 시간 · 토큰 · 비용 | `artifacts/end_to_end.csv` |

In [1]:
from rag.config import ARTIFACTS, load_config
from rag.evaluation.evaluation import (evaluate_end_to_end, evaluate_generation, evaluate_grounding,
                                       evaluate_retrieval)
from rag.usage import usage_summary

cfg = load_config()
ANSWERS = ARTIFACTS / "answers.jsonl"

## 심판 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
이 설정은 네 가지 평가에 공통으로 적용됩니다. `JUDGE = "local"` 이면 API 비용이 없고 아래 `JUDGE_*` 설정은 쓰지 않습니다.
`JUDGE = "ragas"` 일 때만 gpt 심판을 불러 OpenAI API 비용이 발생합니다.


In [2]:
JUDGE = "local"                                 # "local"(API 0원) / "ragas"(gpt 심판, 유료)
JUDGE_MODEL = "gpt-6-luna"                      # 예: "gpt-5-mini"
JUDGE_REASONING_EFFORT = "none"                 # 예: "low"
JUDGE_MAX_TOKENS = cfg["evaluation"]["judge_max_tokens"]             # 출력·추론 토큰 한도
JUDGE_PRICES = cfg["evaluation"]["judge_usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가

cfg["evaluation"].update(judge=JUDGE, judge_model=JUDGE_MODEL, judge_reasoning_effort=JUDGE_REASONING_EFFORT,
                         judge_max_tokens=JUDGE_MAX_TOKENS, judge_usd_per_1m_tokens=JUDGE_PRICES)
cfg["evaluation"]


{'judge': 'local',
 'judge_model': 'gpt-6-luna',
 'judge_reasoning_effort': 'none',
 'judge_max_tokens': 4096,
 'judge_usd_per_1m_tokens': [0.1, 0.01, 0.5]}

## Retrieval
- `context_precision`: 검색한 청크 중 정답에 쓸모 있는 청크가 위에 있나 (local: 정답 문장을 지지하는 청크의 순위 평균 정밀도 · ragas: gpt 판정)
- `context_recall`: 정답의 내용이 검색한 청크에 다 있나 (local: 검색 청크로 지지되는 정답 문장 비율 · ragas: gpt 판정)
- `key_facts_in_context`: 핵심 사실 중 검색한 청크에 들어 있는 비율

local 에서 '지지'는 리랭커가 (문장, 청크)를 관련 있다고 보고(점수 0.5 이상) 문장 속 숫자 · 날짜가 청크에 모두 있다는 뜻입니다.

In [3]:
retrieval = evaluate_retrieval(ANSWERS, cfg)
retrieval

retrieval: 로컬 심판으로 484문항 채점 (API 호출 없음)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


,id,question,context_precision,context_recall,key_facts_in_context
0,numeric-A-night1-r1-0003,한국철도공사의 「운행정보기록 자동분석시스」 사업에서 착수일로부터 사업기간은 얼마인가?,0.0000,0.0000,1.0000
1,numeric-A-night1-r1-0004,한국농어촌공사의 「아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아」 ...,0.6250,1.0000,1.0000
2,numeric-A-night1-r1-0021,을지대학교의 「을지대학교 비교과시스템 개발」 사업에서 교육정보전산시스템의 구축 년도는?,0.0000,0.0000,0.0000
3,numeric-A-night1-r1-0023,한국원자력연구원의 「한국원자력연구원 선량평가시스템 고도화」 사업에서 사업 기간은 얼...,0.0000,0.0000,1.0000
4,numeric-A-night1-r1-0041,한국어촌어항공단의 「한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도」 사업에...,0.1429,1.0000,1.0000
...,...,...,...,...,...
480,needle-36,한국발명진흥회의 「2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편...,1.0000,1.0000,1.0000
481,needle-37,한국산업인력공단의 「RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입 사업」...,1.0000,1.0000,1.0000
482,needle-38,한국수자원공사의 「건설통합시스템(CMS) 고도화」 PSR-001에서 무상 하자보수 ...,1.0000,1.0000,1.0000
483,needle-39,한영대학의 「한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도...,0.8333,1.0000,1.0000


## Generation
- `answer_relevancy`: 답이 질문에 맞는 답인가 (local: 질문과 답의 KURE-v1 임베딩 코사인 유사도 · ragas: 답에서 거꾸로 만든 질문과의 유사도)
- `key_facts_in_answer`: 핵심 사실 중 답에 들어 있는 비율

In [4]:
generation = evaluate_generation(ANSWERS, cfg)
generation

generation: 로컬 심판으로 484문항 채점 (API 호출 없음)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


,id,question,answer_relevancy,key_facts_in_answer
0,numeric-A-night1-r1-0003,한국철도공사의 「운행정보기록 자동분석시스」 사업에서 착수일로부터 사업기간은 얼마인가?,0.6083,1.0000
1,numeric-A-night1-r1-0004,한국농어촌공사의 「아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아」 ...,0.5701,1.0000
2,numeric-A-night1-r1-0021,을지대학교의 「을지대학교 비교과시스템 개발」 사업에서 교육정보전산시스템의 구축 년도는?,0.6295,0.0000
3,numeric-A-night1-r1-0023,한국원자력연구원의 「한국원자력연구원 선량평가시스템 고도화」 사업에서 사업 기간은 얼...,0.5697,1.0000
4,numeric-A-night1-r1-0041,한국어촌어항공단의 「한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도」 사업에...,0.5978,1.0000
...,...,...,...,...
480,needle-36,한국발명진흥회의 「2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편...,0.7383,0.0000
481,needle-37,한국산업인력공단의 「RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입 사업」...,0.6885,0.5000
482,needle-38,한국수자원공사의 「건설통합시스템(CMS) 고도화」 PSR-001에서 무상 하자보수 ...,0.7385,0.6667
483,needle-39,한영대학의 「한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도...,0.6854,1.0000


## Grounding
- `faithfulness`: 답의 주장 중 검색한 청크로 뒷받침되는 비율 — 낮으면 환각 (local: 답 문장 단위 · ragas: gpt 가 나눈 주장 단위)

In [5]:
grounding = evaluate_grounding(ANSWERS, cfg)
grounding

grounding: 로컬 심판으로 484문항 채점 (API 호출 없음)


Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


,id,question,faithfulness
0,numeric-A-night1-r1-0003,한국철도공사의 「운행정보기록 자동분석시스」 사업에서 착수일로부터 사업기간은 얼마인가?,1.0000
1,numeric-A-night1-r1-0004,한국농어촌공사의 「아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아」 ...,1.0000
2,numeric-A-night1-r1-0021,을지대학교의 「을지대학교 비교과시스템 개발」 사업에서 교육정보전산시스템의 구축 년도는?,0.0000
3,numeric-A-night1-r1-0023,한국원자력연구원의 「한국원자력연구원 선량평가시스템 고도화」 사업에서 사업 기간은 얼...,1.0000
4,numeric-A-night1-r1-0041,한국어촌어항공단의 「한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도」 사업에...,1.0000
...,...,...,...
480,needle-36,한국발명진흥회의 「2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편...,1.0000
481,needle-37,한국산업인력공단의 「RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입 사업」...,1.0000
482,needle-38,한국수자원공사의 「건설통합시스템(CMS) 고도화」 PSR-001에서 무상 하자보수 ...,1.0000
483,needle-39,한영대학의 「한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도...,1.0000


## End-to-End
- `factual_correctness`: 답과 정답의 사실 일치 F1 (local: 정답으로 지지되는 답 문장 비율과 답으로 지지되는 정답 문장 비율의 F1 · ragas: gpt 판정)
- `latency_s`: 질문 하나의 재작성 + 검색 + 생성 시간(초)
- `input_tokens` · `cached_input_tokens` · `output_tokens` · `reasoning_tokens`: 답변 생성 토큰 (캐시 입력은 입력, 추론은 출력에 포함)
- `rewrite_*_tokens`: 질문 재작성 토큰
- `answer_cost_usd` · `rewrite_cost_usd`: 호출 당시 단가로 계산해 04에서 저장한 비용
- `cost_usd`: rewrite + 답변 생성 예상 비용 합계. OCR·평가 비용은 `api_usage.csv`에서 별도로 확인하세요.

In [6]:
end_to_end = evaluate_end_to_end(ANSWERS, cfg)
end_to_end

end_to_end: 로컬 심판으로 484문항 채점 (API 호출 없음)


Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


,id,question,factual_correctness,latency_s,input_tokens,output_tokens,cached_input_tokens,reasoning_tokens,rewrite_input_tokens,rewrite_output_tokens,rewrite_cached_input_tokens,rewrite_reasoning_tokens,answer_cost_usd,rewrite_cost_usd,cost_usd
0,numeric-A-night1-r1-0003,한국철도공사의 「운행정보기록 자동분석시스」 사업에서 착수일로부터 사업기간은 얼마인가?,0.0000,7.8600,3736.0000,18.0000,0.0,0.0000,178.0000,114.0000,0.0,75.0000,0.000970,0.000273,0.001242
1,numeric-A-night1-r1-0004,한국농어촌공사의 「아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아」 ...,1.0000,3.9700,3496.0000,21.0000,0.0,0.0000,194.0000,135.0000,0.0,79.0000,0.000916,0.000318,0.001234
2,numeric-A-night1-r1-0021,을지대학교의 「을지대학교 비교과시스템 개발」 사업에서 교육정보전산시스템의 구축 년도는?,0.0000,3.8200,3359.0000,48.0000,0.0,0.0000,176.0000,93.0000,0.0,54.0000,0.000936,0.000230,0.001166
3,numeric-A-night1-r1-0023,한국원자력연구원의 「한국원자력연구원 선량평가시스템 고도화」 사업에서 사업 기간은 얼...,0.0000,2.5000,3926.0000,21.0000,0.0,0.0000,181.0000,56.0000,0.0,14.0000,0.001024,0.000157,0.001181
4,numeric-A-night1-r1-0041,한국어촌어항공단의 「한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도」 사업에...,1.0000,5.3800,3633.0000,26.0000,0.0,0.0000,187.0000,215.0000,0.0,165.0000,0.000960,0.000477,0.001437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,needle-36,한국발명진흥회의 「2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편...,1.0000,4.9600,3750.0000,73.0000,0.0,0.0000,207.0000,112.0000,0.0,43.0000,0.001084,0.000276,0.001359
481,needle-37,한국산업인력공단의 「RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입 사업」...,1.0000,3.9200,3564.0000,35.0000,0.0,0.0000,202.0000,85.0000,0.0,26.0000,0.000961,0.000220,0.001182
482,needle-38,한국수자원공사의 「건설통합시스템(CMS) 고도화」 PSR-001에서 무상 하자보수 ...,0.8000,5.0300,3249.0000,110.0000,0.0,0.0000,199.0000,114.0000,0.0,54.0000,0.001032,0.000278,0.001310
483,needle-39,한영대학의 「한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도...,0.8000,5.2400,3776.0000,113.0000,0.0,0.0000,214.0000,136.0000,0.0,62.0000,0.001170,0.000325,0.001496


## Judge / RAGAS API 사용량 (`JUDGE = "ragas"` 일 때만)
네 평가 함수가 점수 CSV를 자동 저장합니다. RAGAS 내부 Judge 호출은 평가 종류별로 `artifacts/api_usage.csv`에 기록하며, 아래 표는 누적 사용량입니다. RAGAS와 Judge 비용을 중복해서 더하지 않습니다.

In [7]:
usage_summary("judge")

,stage,evaluation,model,reasoning_effort,calls,input_tokens,cached_input_tokens,output_tokens,reasoning_tokens,cost_usd
